Сборка и анализ генома патогенного штамма E. coli

1. Загрузка данных с SRA

In [1]:
prefetch SRR292678
fasterq-dump SRR292678 --split-3 --outdir . --threads 4

prefetch SRR1980037
fasterq-dump SRR1980037 --outdir . --threads 4

SyntaxError: invalid syntax (209512647.py, line 1)

2. Запускаем контроль качества на всех файлах

In [ ]:
fastqc -o . SRR292678_1.fastq SRR292678_2.fastq SRR1980037.fastq

3. Устанавливаем Jellyfish

In [2]:
conda install -c bioconda jellyfish
jellyfish count -C -m 31 -s 5M -t 8 -o k31_counts.jf SRR292678_1.fastq SRR292678_2.fastq
jellyfish histo -o k31_histo.txt k31_counts.jf #гистограмма частот

ValueError: The python kernel does not appear to be a conda environment.  Please use ``%pip install`` instead.

4. Запуск SPAdes

In [ ]:
conda install -c bioconda spades
cd ~/ecoli_x_project
spades.py -1 raw_data/SRR292678_1.fastq -2 raw_data/SRR292678_2.fastq \
          -o assembly/illumina_only -t 8

5. Гибридная сборка (Illumina + PacBio)

In [ ]:
cd ~/ecoli_x_project  # добавляем длинные риды для улучшения качества сборки
spades.py -1 raw_data/SRR292678_1.fastq -2 raw_data/SRR292678_2.fastq \
          --pacbio raw_data/SRR1980037.fastq \
          -o assembly/hybrid -t 8

6. QUAST

In [ ]:
conda install -c bioconda quast
cd ~/ecoli_x_project
quast assembly/illumina_only/scaffolds.fasta assembly/hybrid/scaffolds.fasta \
      -o quast_report -t 8

Ключевые метрики для отчёта:

- N50 — длина контига, при которой 50% генома собрано в контиги не короче этого значения.
- Количество контигов — чем меньше, тем лучше.
- Общая длина сборки — должна быть близка к ожидаемому размеру генома E. coli

7. Аннотация генома с помощью Prokka

In [3]:
conda install -c bioconda prokka
cd ~/ecoli_x_project
prokka assembly/hybrid/scaffolds.fasta --outdir annotation \
        --kingdom Bacteria --genus Escherichia --species coli \
        --strain X --usegenus --addgenes --centre X --compliant --cpus 8
        # запускаем аннотацию на лучшей (гибридной) сборке с флагом --compliant для совместимости с Mauve

SyntaxError: invalid syntax (2331726823.py, line 1)

8. Поиск ближайшего родственника по 16S рРНК

In [ ]:
conda install -c bioconda barrnap
barrnap assembly/hybrid/scaffolds.fasta > annotation/rrna.gff
# извлекаем последовательность 16S рРНК
grep "16S" annotation/rrna.gff | head -1 | \
    awk '{print $1":"$4"-"$5}' > 16s_coords.txt

In [4]:
9. BLAST

SyntaxError: invalid syntax (496265713.py, line 1)

10/\. Поиск шига-токсина: Mauve и анализ
- открываем программу → File → Align with progressiveMauve....
- добавляем файлы: reference.gbk и annotation/scaffolds.gbk.
- запускаем выравнивание.
- используем иконку «Find» (бинокль) для поиска по продукту: shiga toxin.

In [ ]:
11. Анализ устойчивости к антибиотикам (ResFinder)
- поискереходим на сайт ResFinder.
- загружаем файл assembly/hybrid/scaffolds.fasta.
- выбираем «Select Antimicrobial configuration» → «All».
- запускаем поиск
# отчёт записываем, к каким антибиотикам устойчив E. coli X, и к каким — референс